In [86]:
import pandas as pd
cols = ['mean_profile', 'std_profile', 'kurtosis_profile', 'skew_profile', 'mean_dmsnr', 'std_dmsnr', 'kurtosis_dmsnr', 'skew_dmsnr', 'target']
df = pd.read_csv('HTRU_2.csv', header=None, names=cols)
df.head()

,mean_profile,std_profile,kurtosis_profile,skew_profile,mean_dmsnr,std_dmsnr,kurtosis_dmsnr,skew_dmsnr,target
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0


In [87]:
df.shape

(17898, 9)

In [88]:
df.isnull().sum()

mean_profile        0
std_profile         0
kurtosis_profile    0
skew_profile        0
mean_dmsnr          0
std_dmsnr           0
kurtosis_dmsnr      0
skew_dmsnr          0
target              0
dtype: int64

In [89]:
X = df.drop(columns='target')
y = df['target']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [90]:
X_train.shape, X_test.shape

((14318, 8), (3580, 8))

In [91]:
y_train.shape, y_test.shape

((14318,), (3580,))

In [92]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

accuracy_score(y_test, dummy_preds)

0.9083798882681564

In [93]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [94]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_preds = logreg.predict(X_test)

accuracy_score(y_test, logreg_preds)

0.979608938547486

In [95]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

logreg_probs = logreg.predict_proba(X_test)[:,1]

In [96]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (logreg_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    identified = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  identified {identified:4d} / 328 pulsars   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  identified  271 / 328 pulsars   |   missed   57   |   false alarms    16
threshold 0.30:  identified  288 / 328 pulsars   |   missed   40   |   false alarms    26
threshold 0.20:  identified  293 / 328 pulsars   |   missed   35   |   false alarms    38
threshold 0.15:  identified  298 / 328 pulsars   |   missed   30   |   false alarms    48
threshold 0.10:  identified  302 / 328 pulsars   |   missed   26   |   false alarms    87
threshold 0.05:  identified  307 / 328 pulsars   |   missed   21   |   false alarms   214


In [97]:
final_threshold = 0.15
final_preds = (logreg_probs >= final_threshold).astype(int)

print("ROC-AUC", roc_auc_score(y_test, logreg_probs))
print()
print("Model — logistic regression at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

ROC-AUC 0.9709868973689737

Model — logistic regression at threshold 0.15

[[3204   48]
 [  30  298]]

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3252
           1       0.86      0.91      0.88       328

    accuracy                           0.98      3580
   macro avg       0.93      0.95      0.94      3580
weighted avg       0.98      0.98      0.98      3580



In [98]:
feature_names = X.columns
coefficients = logreg.coef_[0]

coef_table = pd.DataFrame({
    'feature': feature_names,
    'weight': coefficients
}).sort_values('weight', ascending=False)

print(coef_table)

            feature    weight
2  kurtosis_profile  6.081413
5         std_dmsnr  0.880606
0      mean_profile  0.575727
1       std_profile -0.125383
6    kurtosis_dmsnr -0.202212
7        skew_dmsnr -0.325565
4        mean_dmsnr -0.826026
3      skew_profile -3.050672


In [99]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:,1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, logreg_probs))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))

Logistic Regression ROC-AUC: 0.9709868973689737
Random Forest ROC-AUC: 0.9621461839618397


In [100]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (rf_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    identified = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  identified {identified:4d} / 328 pulsars   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  identified  287 / 328 pulsars   |   missed   41   |   false alarms    23
threshold 0.30:  identified  295 / 328 pulsars   |   missed   33   |   false alarms    43
threshold 0.20:  identified  300 / 328 pulsars   |   missed   28   |   false alarms    60
threshold 0.15:  identified  300 / 328 pulsars   |   missed   28   |   false alarms    84
threshold 0.10:  identified  302 / 328 pulsars   |   missed   26   |   false alarms   121
threshold 0.05:  identified  305 / 328 pulsars   |   missed   23   |   false alarms   266


In [101]:
final_threshold = 0.20
final_preds = (rf_probs >= final_threshold).astype(int)

print("ROC-AUC", roc_auc_score(y_test, rf_probs))
print()
print("Model — random forest at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

ROC-AUC 0.9621461839618397

Model — random forest at threshold 0.2

[[3192   60]
 [  28  300]]

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3252
           1       0.83      0.91      0.87       328

    accuracy                           0.98      3580
   macro avg       0.91      0.95      0.93      3580
weighted avg       0.98      0.98      0.98      3580



In [102]:
feature_names = X.columns
importance = rf.feature_importances_

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print(importances)

            feature  importance
2  kurtosis_profile    0.369460
3      skew_profile    0.212779
0      mean_profile    0.144158
4        mean_dmsnr    0.071431
5         std_dmsnr    0.062132
1       std_profile    0.050621
7        skew_dmsnr    0.049876
6    kurtosis_dmsnr    0.039544


In [103]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
gb_probs = gb.predict_proba(X_test)[:,1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, logreg_probs))
print("Random Forest ROC-AUC:      ", roc_auc_score(y_test, rf_probs))
print("Gradient Boosting ROC-AUC:  ", roc_auc_score(y_test, gb_probs))

Logistic Regression ROC-AUC: 0.9709868973689737
Random Forest ROC-AUC:       0.9621461839618397
Gradient Boosting ROC-AUC:   0.9727414461644617


In [104]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (gb_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    identified = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  identified {identified:4d} / 328 pulsars   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  identified  286 / 328 pulsars   |   missed   42   |   false alarms    25
threshold 0.30:  identified  290 / 328 pulsars   |   missed   38   |   false alarms    43
threshold 0.20:  identified  295 / 328 pulsars   |   missed   33   |   false alarms    58
threshold 0.15:  identified  296 / 328 pulsars   |   missed   32   |   false alarms    67
threshold 0.10:  identified  297 / 328 pulsars   |   missed   31   |   false alarms    77
threshold 0.05:  identified  299 / 328 pulsars   |   missed   29   |   false alarms   122


In [105]:
final_threshold = 0.20
final_preds = (gb_probs >= final_threshold).astype(int)

print("ROC-AUC", roc_auc_score(y_test, gb_probs))
print()
print("Model — gradient boosting at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

ROC-AUC 0.9727414461644617

Model — gradient boosting at threshold 0.2

[[3194   58]
 [  33  295]]

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3252
           1       0.84      0.90      0.87       328

    accuracy                           0.97      3580
   macro avg       0.91      0.94      0.93      3580
weighted avg       0.98      0.97      0.97      3580



In [106]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    gb, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

            feature  importance
2  kurtosis_profile    0.434906
5         std_dmsnr    0.014623
3      skew_profile    0.003012
1       std_profile    0.000805
7        skew_dmsnr    0.000691
6    kurtosis_dmsnr   -0.000562
0      mean_profile   -0.000627
4        mean_dmsnr   -0.001304


In [107]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

            feature  importance
2  kurtosis_profile    0.100573
3      skew_profile    0.003611
5         std_dmsnr    0.000303
4        mean_dmsnr   -0.000670
6    kurtosis_dmsnr   -0.001221
7        skew_dmsnr   -0.001783
0      mean_profile   -0.002047
1       std_profile   -0.005201


In [108]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    logreg, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

            feature  importance
2  kurtosis_profile    0.788295
3      skew_profile    0.016118
5         std_dmsnr    0.011750
4        mean_dmsnr    0.004466
1       std_profile    0.001257
0      mean_profile   -0.000767
6    kurtosis_dmsnr   -0.001681
7        skew_dmsnr   -0.002392
